# 01 — Exploratory Data Analysis: Market Regime Evidence

> **Goal:** Visually confirm that distinct market regimes exist in S&P 500 data  
> *before* committing to a Hidden Markov Model. We look for volatility clustering,  
> fat tails in the return distribution, and separable clusters in feature space.

**Sections**
1. S&P 500 price history with known crisis periods
2. Return distribution — histogram + KDE, skew & kurtosis
3. Rolling realised volatility
4. Volume ratio
5. Feature scatter: log return vs. realised volatility
6. ACF of squared returns — ARCH effect test
7. Summary statistics table

In [ ]:
import sys
import warnings
from pathlib import Path

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import yaml

from src.data.fetch import fetch_ohlcv
from src.features.engineer import build_feature_matrix

# ── aesthetics ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})

BLUE   = "#2563EB"
INDIGO = "#6366F1"
SLATE  = "#94A3B8"
CRISIS = "#FCA5A5"   # soft red for crisis shading

# ── config & data ──────────────────────────────────────────────────────────
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

df = fetch_ohlcv(
    config["data"]["ticker"],
    start=config["data"]["start_date"],
    end=config["data"]["end_date"],
)
features = build_feature_matrix(df, config)

print(f"OHLCV    : {df.shape}  [{df.index[0].date()} → {df.index[-1].date()}]")
print(f"Features : {features.shape}")
features.head()

---
## 1. S&P 500 Price History
Shaded bands mark the three major drawdown episodes in the sample period.

In [ ]:
CRISES = [
    ("2007-10-09", "2009-03-09", "2008 GFC"),
    ("2020-02-19", "2020-03-23", "COVID-19"),
    ("2022-01-03", "2022-10-12", "2022 Bear"),
]

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df.index, df["Close"], color=BLUE, lw=1.3, label="S&P 500 Close")

for start, end, label in CRISES:
    ax.axvspan(
        pd.Timestamp(start), pd.Timestamp(end),
        color=CRISIS, alpha=0.45, label=label,
    )

ax.set_title("S&P 500 — Daily Close Price (2000–present)")
ax.set_xlabel("")
ax.set_ylabel("Price (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(loc="upper left", framealpha=0.85, fontsize=10)

plt.tight_layout()
plt.savefig("../outputs/01_price_history.png", bbox_inches="tight")
plt.show()

---
## 2. Return Distribution

A normal distribution would be a thin, symmetric bell curve. Real market returns show  
**negative skew** (large drops more common than large rallies) and **excess kurtosis**  
(fat tails — extreme events occur far more often than Gaussian theory predicts).

In [ ]:
ret   = features["log_return"]
skew_ = stats.skew(ret)
kurt_ = stats.kurtosis(ret, fisher=True)   # excess (Fisher) kurtosis

fig, ax = plt.subplots(figsize=(9, 4.5))

sns.histplot(
    ret, bins=120, kde=True, stat="density",
    color=BLUE, alpha=0.50, ax=ax,
    line_kws={"lw": 2.2, "label": "KDE"},
)

# Normal reference
x = np.linspace(ret.min(), ret.max(), 500)
ax.plot(
    x, stats.norm.pdf(x, ret.mean(), ret.std()),
    "--", color="tomato", lw=1.8, label="Normal reference",
)

ax.set_title("Daily Log-Return Distribution")
ax.set_xlabel("Log Return")
ax.set_ylabel("Density")
ax.legend(framealpha=0.85)

ax.text(
    0.975, 0.93,
    f"Skewness      = {skew_:+.3f}\nExcess kurtosis = {kurt_:.2f}",
    transform=ax.transAxes, ha="right", va="top", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CBD5E1", alpha=0.9),
)

plt.tight_layout()
plt.savefig("../outputs/02_return_distribution.png", bbox_inches="tight")
plt.show()

print(f"Skewness      : {skew_:+.4f}")
print(f"Excess kurtosis: {kurt_:.4f}")

> **Insight — Fat tails:** Excess kurtosis >> 0 confirms the distribution has heavier tails  
> than a Gaussian. A Gaussian emission HMM is therefore an *approximation*, but it remains  
> the standard baseline for regime detection and is sufficient for our purposes.

---
## 3. Rolling Realised Volatility

If the market were in a single, stationary regime, volatility would be roughly constant.  
Instead we see prolonged **high-vol** and **low-vol** episodes — the textbook signature of hidden states.

In [ ]:
vol_pct = features["realized_vol"] * 100

fig, ax = plt.subplots(figsize=(14, 3.8))

ax.fill_between(features.index, vol_pct, alpha=0.30, color=BLUE)
ax.plot(features.index, vol_pct, lw=1.1, color=BLUE, label="Realised Vol")

for start, end, label in CRISES:
    ax.axvspan(
        pd.Timestamp(start), pd.Timestamp(end),
        color=CRISIS, alpha=0.35, label=label,
    )

ax.axhline(vol_pct.mean(), color="black", lw=0.9, ls=":",
           label=f"Mean = {vol_pct.mean():.1f}%")

ax.set_title("21-Day Realised Volatility — Annualised (%)")
ax.set_xlabel("")
ax.set_ylabel("Volatility (%)")
ax.legend(loc="upper right", framealpha=0.85, fontsize=10)

plt.tight_layout()
plt.savefig("../outputs/03_rolling_volatility.png", bbox_inches="tight")
plt.show()

> **Insight — Volatility clusters:** Volatility is clearly not i.i.d. — it persists in  
> high and low regimes for months at a time. This temporal clustering is the primary  
> motivation for a **hidden-state model**: each state captures a volatility regime.

---
## 4. Volume Ratio

Elevated volume (ratio > 1) during drawdowns confirms that crisis regimes are characterised  
by *both* high volatility and above-average trading activity.

In [ ]:
vr = features["volume_ratio"]

fig, ax = plt.subplots(figsize=(14, 3))

ax.fill_between(
    features.index, vr, 1.0,
    where=(vr >= 1.0), alpha=0.50, color=INDIGO, label="Above avg volume",
)
ax.fill_between(
    features.index, vr, 1.0,
    where=(vr < 1.0), alpha=0.40, color=SLATE, label="Below avg volume",
)
ax.plot(features.index, vr, lw=0.6, color="#475569", alpha=0.7)
ax.axhline(1.0, color="black", lw=0.9, ls="--", label="Rolling mean")

ax.set_title("Volume Ratio (Daily / 21-Day Rolling Mean)")
ax.set_xlabel("")
ax.set_ylabel("Ratio")
ax.legend(framealpha=0.85, fontsize=10)

plt.tight_layout()
plt.savefig("../outputs/04_volume_ratio.png", bbox_inches="tight")
plt.show()

---
## 5. Feature Scatter: Log Return vs. Realised Volatility

If no regimes existed, we would see a single undifferentiated cloud.  
Distinct clusters here mean the HMM has separable states to find.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

sc = ax.scatter(
    features["log_return"] * 100,
    features["realized_vol"] * 100,
    c=features["volume_ratio"],
    cmap="viridis", alpha=0.30, s=7, linewidths=0,
)
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Volume Ratio", fontsize=10)

ax.set_title("Log Return vs. Realised Volatility\n(colour = volume ratio)")
ax.set_xlabel("Log Return (%)")
ax.set_ylabel("Realised Vol — Annualised (%)")

plt.tight_layout()
plt.savefig("../outputs/05_scatter.png", bbox_inches="tight")
plt.show()

> **Insight — Distinct scatter clusters:** The scatter shows at least two visually separable  
> regions: a dense low-vol cluster (most trading days) and a high-vol / wide-return tail  
> (crisis periods). The HMM should recover these as distinct hidden states.

---
## 6. Autocorrelation of Squared Returns — ARCH Effect

Raw returns are roughly uncorrelated (no easy edge), but **squared returns** exhibit  
strong positive autocorrelation. This is the ARCH effect — direct statistical evidence  
that volatility is serially dependent and therefore modelable.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(
    features["log_return"], lags=50, alpha=0.05,
    ax=axes[0], title="ACF — Log Returns",
    color=BLUE, vlines_kwargs={"colors": BLUE},
)
plot_acf(
    features["log_return"] ** 2, lags=50, alpha=0.05,
    ax=axes[1], title="ACF — Squared Returns (ARCH effect)",
    color="tomato", vlines_kwargs={"colors": "tomato"},
)

for ax in axes:
    ax.set_xlabel("Lag (days)")
    ax.set_ylabel("Autocorrelation")

plt.suptitle(
    "Serial dependence: returns are uncorrelated, variance is not",
    y=1.02, fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.savefig("../outputs/06_acf.png", bbox_inches="tight")
plt.show()

> **Insight — Volatility clustering confirmed:** Near-zero ACF on raw returns rules out  
> simple momentum. Significant ACF on *squared* returns through lag 50+ is a direct  
> empirical argument for a **hidden state model** over i.i.d. assumptions.

---
## 7. Summary Statistics

In [ ]:
from scipy.stats import skew, kurtosis

summary = features.agg(["mean", "std", "min", "max"]).T
summary.columns = ["Mean", "Std", "Min", "Max"]
summary["Skewness"]     = features.apply(skew)
summary["Ex. Kurtosis"] = features.apply(kurtosis)   # excess (Fisher)

summary.index = ["log_return", "realized_vol", "volume_ratio"]
summary.round(4)

---
## Key Insights

| # | Observation | Implication for modelling |
|---|-------------|---------------------------|
| 1 | **Volatility clusters** — vol persists in high and low regimes for months | Supports a hidden-state model; each state captures a vol regime |
| 2 | **Fat tails** — excess kurtosis >> 0, heavier than Gaussian | Gaussian emission is an approximation, but standard for HMM; consider t-emission as a future extension |
| 3 | **Distinct scatter clusters** — low-vol dense cloud + high-vol tail | HMM should recover these as separate hidden states |
| 4 | **ARCH effect** — ACF of squared returns significant through lag 50+ | Variance is serially dependent → directly justifies a latent-regime model over i.i.d. |
| 5 | **Volume spikes during crises** — ratio > 1 during GFC / COVID / 2022 | Volume ratio adds discriminatory power beyond price alone; include in feature set |

---
*All plots saved to `outputs/`. Proceed to `02_model_selection.ipynb` for BIC sweep over number of states.*